# 01 — IndicConformer 600M ASR — Clean Live Notebook

**Goal:** Hindi/Urdu speech → accurate transcript.

**Pipeline:** Microphone → 16 kHz normalization → RMS VAD → unlimited utterance accumulation → 5 seconds of continuous silence → IndicConformer CTC/RNNT → persistent WAV + JSONL/CSV audit.

There is **no maximum speech duration**. CTC and RNNT remain selectable for later comparison.

This is the cleaned notebook: one model load, one ASR function, one VAD+ASR callback, one Gradio app.

In [1]:
# 0. Verify Colab GPU
import sys
import time
import json
import csv
import traceback
from pathlib import Path
from datetime import datetime, timezone

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable a GPU in Colab: Runtime → Change runtime type → GPU"
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    f"VRAM: "
    f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
)

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


In [2]:
# 1. Install dependencies
!pip -q install -U transformers torchaudio soundfile gradio huggingface_hub onnxruntime

print("✅ Dependencies installed")

✅ Dependencies installed


In [3]:
# 2. Hugging Face authentication + model loading
from huggingface_hub import login

login()

In [4]:
from transformers import AutoModel

MODEL_ID = "ai4bharat/indic-conformer-600m-multilingual"
DEVICE = "cuda"

print("Loading model...")
t0 = time.perf_counter()

model = AutoModel.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
).to(DEVICE)

model.eval()

print(
    f"✅ Loaded in "
    f"{time.perf_counter() - t0:.2f}s"
)

Loading model...


config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

model_onnx.py:   0%|          | 0.00/9.64k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual:
- model_onnx.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate


/usr/local/lib/python3.13/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


✅ Loaded in 51.34s


In [5]:
# 3. Imports + configuration
import numpy as np
import torch
import torchaudio
import soundfile as sf
import gradio as gr

# MODEL_ID = "ai4bharat/indic-conformer-600m-multilingual"
# DEVICE = "cuda"
TARGET_SR = 16000

LANGUAGES = {
    "Hindi": "hi",
    "Urdu": "ur",
}

# VAD policy
VAD_THRESHOLD = 0.005
SILENCE_CONFIRM_SECONDS = 5.0
MIN_UTTERANCE_SECONDS = 0.5
# Keep one second of audio before VAD detects speech.
PRE_ROLL_SECONDS = 0.0

print("Model:", MODEL_ID)
print("Device:", DEVICE)
print("Target sample rate:", TARGET_SR)
print("VAD threshold:", VAD_THRESHOLD)
print("Silence required:", SILENCE_CONFIRM_SECONDS, "s")
print("Maximum speech duration: UNLIMITED")

Model: ai4bharat/indic-conformer-600m-multilingual
Device: cuda
Target sample rate: 16000
VAD threshold: 0.005
Silence required: 5.0 s
Maximum speech duration: UNLIMITED


In [6]:
# 4. Known-good file loader
def load_audio(path: str):
    wav, sr = torchaudio.load(path)

    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    if sr != TARGET_SR:
        wav = torchaudio.transforms.Resample(
            sr,
            TARGET_SR,
        )(wav)
        sr = TARGET_SR

    return wav, sr

print("✅ Known-good loader ready")

✅ Known-good loader ready


In [7]:
# 5. Exact IndicConformer ASR call
# Keep this call stable while we benchmark CTC vs RNNT.

def run_indicconformer(
    audio,
    language_name,
    decoder_name,
):
    if audio is None or len(audio) == 0:
        return "", 0.0

    lang = LANGUAGES[language_name]

    input_wav = (
        torch.from_numpy(audio)
        .float()
        .unsqueeze(0)
        .to(DEVICE)
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    duration = len(audio) / TARGET_SR

    print(
        f"[ASR] START | "
        f"language={lang} | "
        f"decoder={decoder_name} | "
        f"audio={duration:.2f}s"
    )

    t0 = time.perf_counter()

    with torch.inference_mode():
        text = model(
            input_wav,
            lang,
            decoder_name.lower(),
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0

    if isinstance(text, (list, tuple)):
        text = text[0]

    text = str(text).strip()

    rtf = elapsed / duration if duration > 0 else 0.0

    print(f"[ASR] RESULT: {text!r}")
    print(f"[ASR] TIME: {elapsed:.3f}s | RTF={rtf:.3f}")

    return text, elapsed

In [8]:
# 6. Audio normalization + RMS

def normalize_audio(audio):
    if audio is None:
        return None

    sr, data = audio
    data = np.asarray(data)

    print(
        f"\n[AUDIO] incoming: "
        f"{sr} {data.shape} {data.dtype}"
    )

    if data.ndim > 1:
        data = data.mean(axis=1)

    if np.issubdtype(data.dtype, np.integer):
        info = np.iinfo(data.dtype)
        scale = max(abs(info.min), info.max)
        data = data.astype(np.float32) / scale
    else:
        data = data.astype(np.float32)

    if sr != TARGET_SR:
        old_len = len(data)
        new_len = int(round(old_len * TARGET_SR / sr))

        if old_len and new_len:
            old_x = np.linspace(0.0, 1.0, old_len)
            new_x = np.linspace(0.0, 1.0, new_len)
            data = np.interp(
                new_x,
                old_x,
                data,
            ).astype(np.float32)

        print(
            f"[AUDIO] resampled "
            f"{sr} → {TARGET_SR}"
        )

    print(
        f"[AUDIO] normalized: "
        f"{len(data) / TARGET_SR:.3f}s"
    )

    return data


def calculate_rms(audio):
    if audio is None or len(audio) == 0:
        return 0.0

    audio = audio.astype(np.float32)

    return float(
        np.sqrt(
            np.mean(audio ** 2)
        )
    )

print("✅ Audio utilities ready")

✅ Audio utilities ready


In [9]:
# 7. Persistent audit storage

AUDIT_DIR = Path("/content/asr_audit")
AUDIO_DIR = AUDIT_DIR / "audio"

AUDIT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

SESSION_ID = datetime.now(
    timezone.utc
).strftime("%Y%m%dT%H%M%SZ")

JSONL_PATH = AUDIT_DIR / "utterance_log.jsonl"
CSV_PATH = AUDIT_DIR / "utterance_log.csv"

print("✅ Audit storage ready")
print("Session ID:", SESSION_ID)
print("Audit directory:", AUDIT_DIR)

✅ Audit storage ready
Session ID: 20260906T091947Z
Audit directory: /content/asr_audit


In [10]:
# 8. Persistent audit writer

def save_asr_audit(
    utterance_number,
    audio,
    text,
    language_name,
    decoder_name,
    speech_duration,
    inference_time,
    status,
    error=None,
):
    timestamp = datetime.now(
        timezone.utc
    ).isoformat()

    audio_duration = (
        len(audio) / TARGET_SR
        if audio is not None
        else 0.0
    )

    audio_path = (
        AUDIO_DIR
        / f"utterance_{utterance_number:04d}.wav"
    )

    record = {
        "session_id": SESSION_ID,
        "utterance_id": utterance_number,
        "timestamp_utc": timestamp,
        "language": language_name,
        "language_code": LANGUAGES[language_name],
        "decoder": decoder_name.upper(),
        "speech_duration_s": round(
            speech_duration, 3
        ),
        "asr_audio_duration_s": round(
            audio_duration, 3
        ),
        "inference_time_s": (
            round(inference_time, 3)
            if inference_time is not None
            else None
        ),
        "rtf": (
            round(
                inference_time / audio_duration,
                4,
            )
            if inference_time is not None
            and audio_duration > 0
            else None
        ),
        "transcript": text,
        "status": status,
        "error": error,
        "audio_file": str(audio_path),
    }

    with JSONL_PATH.open(
        "a",
        encoding="utf-8",
    ) as f:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

    csv_exists = CSV_PATH.exists()

    with CSV_PATH.open(
        "a",
        encoding="utf-8-sig",
        newline="",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=list(record.keys()),
        )

        if not csv_exists:
            writer.writeheader()

        writer.writerow(record)

    print(
        f"[AUDIT] Metadata saved for "
        f"utterance {utterance_number}"
    )

    return record

In [11]:
# 9. Per-session state

def make_initial_state():

    return {
        "speech_active": False,

        "speech_duration": 0.0,
        "silence_duration": 0.0,

        # Audio captured BEFORE speech detection.
        # Used to protect the beginning of an utterance.
        "pre_roll_audio": [],

        # Audio captured from speech start onward.
        "utterance_audio": [],

        "utterance_count": 0,
        "transcript": "",
        "total_speech_seconds": 0.0,
    }

print("✅ State factory ready")

✅ State factory ready


In [12]:
# 10. VAD → complete utterance → ASR → audit

def live_vad_asr(
    audio,
    state,
    language_name,
    decoder_name,
):
    if state is None:
        state = make_initial_state()

    try:
        if audio is None:
            return (
                state["transcript"]
                or "🎙 Waiting for microphone...",
                state,
            )

        chunk = normalize_audio(audio)

        if chunk is None or len(chunk) == 0:
            return (
                state["transcript"]
                or "⚠️ Empty audio",
                state,
            )

        chunk_duration = len(chunk) / TARGET_SR
        rms = calculate_rms(chunk)
        is_speech = rms >= VAD_THRESHOLD

        print("\n" + "=" * 72)
        print(
            f"[AUDIO] "
            f"{chunk_duration:.2f}s | "
            f"RMS={rms:.6f}"
        )

        # ----------------------------------------------------
        # SPEECH
        # ----------------------------------------------------
        # ----------------------------------------------------
        # SPEECH
        # ----------------------------------------------------

        if is_speech:

            if not state["speech_active"]:

                print("🟢 SPEECH START")

                state["speech_active"] = True
                state["speech_duration"] = 0.0
                state["silence_duration"] = 0.0

                # ------------------------------------------------
                # PRE-ROLL
                # ------------------------------------------------
                # Include the most recent idle audio so that
                # the beginning of the first word/sentence is
                # not lost.

                state["utterance_audio"] = (
                    state["pre_roll_audio"].copy()
                )

                state["pre_roll_audio"] = []


            # Speech cancels silence.
            state["silence_duration"] = 0.0


            # Add current speech chunk.
            state["utterance_audio"].extend(
                chunk.tolist()
            )


            state["speech_duration"] += (
                chunk_duration
            )

        # ----------------------------------------------------
        # SILENCE
        # ----------------------------------------------------

        # ----------------------------------------------------
        # SILENCE
        # ----------------------------------------------------

        else:

            if state["speech_active"]:

                # Silence inside an active utterance.
                state["silence_duration"] += (
                    chunk_duration
                )

                # Keep temporarily.
                # Confirmed trailing silence is removed
                # when the utterance is finalized.

                state["utterance_audio"].extend(
                    chunk.tolist()
                )

            else:

                # ------------------------------------------------
                # IDLE PRE-ROLL
                # ------------------------------------------------
                # Keep the most recent PRE_ROLL_SECONDS of audio
                # while waiting for speech.

                state["pre_roll_audio"].extend(
                    chunk.tolist()
                )

                max_pre_roll = int(
                    PRE_ROLL_SECONDS * TARGET_SR
                )

                if len(state["pre_roll_audio"]) > max_pre_roll:

                    state["pre_roll_audio"] = (
                        state["pre_roll_audio"][
                            -max_pre_roll:
                        ]
                    )

        # ----------------------------------------------------
        # FINALIZE AFTER 5 SECONDS CONTINUOUS SILENCE
        # ----------------------------------------------------
        utterance_complete = (
            state["speech_active"]
            and state["speech_duration"]
            >= MIN_UTTERANCE_SECONDS
            and state["silence_duration"]
            >= SILENCE_CONFIRM_SECONDS
        )

        if utterance_complete:

            state["utterance_count"] += 1
            utterance_number = state["utterance_count"]

            print("\n" + "#" * 72)
            print(
                f"[UTTERANCE {utterance_number}] COMPLETE"
            )
            print(
                f"Speech duration: "
                f"{state['speech_duration']:.2f}s"
            )
            print(
                f"Confirmed silence: "
                f"{state['silence_duration']:.2f}s"
            )

            audio_to_transcribe = np.asarray(
                state["utterance_audio"],
                dtype=np.float32,
            )

            # Remove confirmed trailing silence.
            silence_samples = int(
                state["silence_duration"]
                * TARGET_SR
            )

            if (
                silence_samples > 0
                and silence_samples < len(
                    audio_to_transcribe
                )
            ):
                audio_to_transcribe = (
                    audio_to_transcribe[
                        :-silence_samples
                    ]
                )

            asr_duration = (
                len(audio_to_transcribe)
                / TARGET_SR
            )

            print(
                f"Audio sent to ASR: "
                f"{asr_duration:.2f}s"
            )

            # ------------------------------------------------
            # Save exact ASR input BEFORE inference.
            # ------------------------------------------------
            audio_path = (
                AUDIO_DIR
                / f"utterance_{utterance_number:04d}.wav"
            )

            sf.write(
                str(audio_path),
                audio_to_transcribe,
                TARGET_SR,
            )

            print(
                f"[AUDIT] Audio saved: "
                f"{audio_path}"
            )

            text = ""
            inference_time = None
            error_text = None
            status = "error"

            # ------------------------------------------------
            # ASR
            # ------------------------------------------------
            try:
                text, inference_time = (
                    run_indicconformer(
                        audio_to_transcribe,
                        language_name,
                        decoder_name,
                    )
                )
                status = "ok"

            except Exception as e:
                error_text = (
                    f"{type(e).__name__}: {e}"
                )

                print(
                    f"[UTTERANCE "
                    f"{utterance_number}] "
                    f"❌ ASR FAILED"
                )

                traceback.print_exc()

            # ------------------------------------------------
            # Persist metadata.
            # ------------------------------------------------
            save_asr_audit(
                utterance_number=utterance_number,
                audio=audio_to_transcribe,
                text=(
                    text
                    if status == "ok"
                    else None
                ),
                language_name=language_name,
                decoder_name=decoder_name,
                speech_duration=(
                    state["speech_duration"]
                ),
                inference_time=inference_time,
                status=status,
                error=error_text,
            )

            # ------------------------------------------------
            # Commit result.
            # ------------------------------------------------
            if status == "ok" and text:

                state["transcript"] += (
                    f"[Utterance "
                    f"{utterance_number}] "
                    f"{text}\n"
                )

                print(
                    f"[UTTERANCE "
                    f"{utterance_number}] "
                    f"✅ COMMITTED"
                )

            elif status == "ok":

                state["transcript"] += (
                    f"[Utterance "
                    f"{utterance_number}] "
                    f"⚠️ ASR returned empty\n"
                )

                print(
                    f"[UTTERANCE "
                    f"{utterance_number}] "
                    f"⚠️ EMPTY ASR"
                )

            else:

                state["transcript"] += (
                    f"[Utterance "
                    f"{utterance_number}] "
                    f"❌ ASR ERROR\n"
                )

            state["total_speech_seconds"] += (
                state["speech_duration"]
            )

            # ------------------------------------------------
            # ALWAYS RESET
            # ------------------------------------------------
            state["speech_active"] = False
            state["speech_duration"] = 0.0
            state["silence_duration"] = 0.0
            state["utterance_audio"] = []
            state["pre_roll_audio"] = []
            print(
                f"[UTTERANCE "
                f"{utterance_number}] "
                f"STATE RESET"
            )
            print("#" * 72)

        # ----------------------------------------------------
        # UI status
        # ----------------------------------------------------
        if state["speech_active"]:

            status_line = (
                "\n\n🟢 SPEAKING\n"
                f"Speech: "
                f"{state['speech_duration']:.1f}s\n"
                f"Silence: "
                f"{state['silence_duration']:.1f}/"
                f"{SILENCE_CONFIRM_SECONDS:.1f}s"
            )

        else:

            status_line = (
                "\n\n⚪ LISTENING\n"
                f"Utterances completed: "
                f"{state['utterance_count']}"
            )

        return (
            state["transcript"]
            + status_line,
            state,
        )

    except Exception as e:

        print("\n❌ LIVE CALLBACK ERROR")
        traceback.print_exc()

        return (
            state.get("transcript", "")
            + f"\n\n❌ ERROR: "
            f"{type(e).__name__}: {e}",
            state,
        )

The following cell was drafted on 6/9/2026 at 2:54PM!! It has better interfce than previous.The previous cell is below

In [16]:
# ============================================================
# 10. Production Live VAD → ASR → Audit Callback
# ============================================================

def live_vad_asr(
    audio,
    state,
    language_name,
    decoder_name,
):
    if state is None:
        state = make_initial_state()

    def ui_status(state, mode="idle"):
        count = state["utterance_count"]
        speech = state["speech_duration"]
        silence = state["silence_duration"]

        if mode == "speaking":
            silence_pct = min(
                100,
                (silence / SILENCE_CONFIRM_SECONDS) * 100
            )

            return f"""
            <div class="status-card speaking">
                <div class="status-top">
                    <div class="status-indicator"></div>
                    <div>
                        <div class="status-title">Listening</div>
                        <div class="status-subtitle">
                            Speech detected
                        </div>
                    </div>
                </div>

                <div class="status-metrics">
                    <div class="metric">
                        <span class="metric-label">Speech</span>
                        <span class="metric-value">
                            {speech:.1f}s
                        </span>
                    </div>

                    <div class="metric">
                        <span class="metric-label">Silence</span>
                        <span class="metric-value">
                            {silence:.1f}s
                        </span>
                    </div>

                    <div class="metric">
                        <span class="metric-label">Utterances</span>
                        <span class="metric-value">
                            {count}
                        </span>
                    </div>
                </div>

                <div class="silence-track">
                    <div
                        class="silence-progress"
                        style="width:{silence_pct}%"
                    ></div>
                </div>

                <div class="silence-caption">
                    Utterance completes after
                    {SILENCE_CONFIRM_SECONDS:.0f}s of continuous silence
                </div>
            </div>
            """

        elif mode == "processing":

            return f"""
            <div class="status-card processing">
                <div class="status-top">
                    <div class="status-indicator"></div>
                    <div>
                        <div class="status-title">
                            Processing
                        </div>
                        <div class="status-subtitle">
                            Transcribing utterance
                        </div>
                    </div>
                </div>

                <div class="processing-line">
                    IndicConformer is processing your speech...
                </div>
            </div>
            """

        elif mode == "error":

            return f"""
            <div class="status-card error">
                <div class="status-top">
                    <div class="status-indicator"></div>
                    <div>
                        <div class="status-title">
                            Processing error
                        </div>
                        <div class="status-subtitle">
                            The session remains active
                        </div>
                    </div>
                </div>
            </div>
            """

        else:

            return f"""
            <div class="status-card ready">
                <div class="status-top">
                    <div class="status-indicator"></div>
                    <div>
                        <div class="status-title">
                            Ready
                        </div>
                        <div class="status-subtitle">
                            Speak naturally — transcription will appear here
                        </div>
                    </div>
                </div>

                <div class="status-metrics">
                    <div class="metric">
                        <span class="metric-label">Utterances</span>
                        <span class="metric-value">
                            {count}
                        </span>
                    </div>

                    <div class="metric">
                        <span class="metric-label">Speech captured</span>
                        <span class="metric-value">
                            {state["total_speech_seconds"]:.1f}s
                        </span>
                    </div>
                </div>
            </div>
            """

    try:

        if audio is None:
            return (
                state["transcript"],
                ui_status(state, "idle"),
                state,
            )

        chunk = normalize_audio(audio)

        if chunk is None or len(chunk) == 0:
            return (
                state["transcript"],
                ui_status(state, "idle"),
                state,
            )

        chunk_duration = len(chunk) / TARGET_SR
        rms = calculate_rms(chunk)
        is_speech = rms >= VAD_THRESHOLD

        # --------------------------------------------------------
        # Internal engineering log
        # --------------------------------------------------------

        print("\n" + "=" * 72)
        print(
            f"[AUDIO] "
            f"{chunk_duration:.2f}s | "
            f"RMS={rms:.6f}"
        )

        # ========================================================
        # SPEECH
        # ========================================================

        if is_speech:

            if not state["speech_active"]:

                print("[VAD] SPEECH START")

                state["speech_active"] = True
                state["speech_duration"] = 0.0
                state["silence_duration"] = 0.0

                state["utterance_audio"] = (
                    state["pre_roll_audio"].copy()
                )

                state["pre_roll_audio"] = []

            state["silence_duration"] = 0.0

            state["utterance_audio"].extend(
                chunk.tolist()
            )

            state["speech_duration"] += chunk_duration

        # ========================================================
        # SILENCE
        # ========================================================

        else:

            if state["speech_active"]:

                state["silence_duration"] += chunk_duration

                state["utterance_audio"].extend(
                    chunk.tolist()
                )

            else:

                state["pre_roll_audio"].extend(
                    chunk.tolist()
                )

                max_pre_roll = int(
                    PRE_ROLL_SECONDS * TARGET_SR
                )

                if len(state["pre_roll_audio"]) > max_pre_roll:

                    state["pre_roll_audio"] = (
                        state["pre_roll_audio"][
                            -max_pre_roll:
                        ]
                    )

        # ========================================================
        # FINALIZE AFTER CONTINUOUS SILENCE
        # ========================================================

        utterance_complete = (
            state["speech_active"]
            and state["speech_duration"]
            >= MIN_UTTERANCE_SECONDS
            and state["silence_duration"]
            >= SILENCE_CONFIRM_SECONDS
        )

        if utterance_complete:

            state["utterance_count"] += 1

            utterance_number = state["utterance_count"]

            print(
                f"[UTTERANCE {utterance_number}] "
                f"COMPLETE"
            )

            audio_to_transcribe = np.asarray(
                state["utterance_audio"],
                dtype=np.float32,
            )

            # ----------------------------------------------------
            # Remove confirmed trailing silence
            # ----------------------------------------------------

            silence_samples = int(
                state["silence_duration"]
                * TARGET_SR
            )

            if (
                silence_samples > 0
                and silence_samples < len(
                    audio_to_transcribe
                )
            ):
                audio_to_transcribe = (
                    audio_to_transcribe[
                        :-silence_samples
                    ]
                )

            asr_duration = (
                len(audio_to_transcribe)
                / TARGET_SR
            )

            # ----------------------------------------------------
            # Save exact ASR input
            # ----------------------------------------------------

            audio_path = (
                AUDIO_DIR
                / f"utterance_{utterance_number:04d}.wav"
            )

            sf.write(
                str(audio_path),
                audio_to_transcribe,
                TARGET_SR,
            )

            print(
                f"[AUDIT] Audio saved: "
                f"{audio_path}"
            )

            text = ""
            inference_time = None
            error_text = None
            status = "error"

            # ----------------------------------------------------
            # ASR
            # ----------------------------------------------------

            try:

                text, inference_time = (
                    run_indicconformer(
                        audio_to_transcribe,
                        language_name,
                        decoder_name,
                    )
                )

                status = "ok"

            except Exception as e:

                error_text = (
                    f"{type(e).__name__}: {e}"
                )

                print(
                    f"[UTTERANCE {utterance_number}] "
                    f"ASR FAILED"
                )

                traceback.print_exc()

            # ----------------------------------------------------
            # Persist metadata
            # ----------------------------------------------------

            save_asr_audit(
                utterance_number=utterance_number,
                audio=audio_to_transcribe,
                text=(
                    text
                    if status == "ok"
                    else None
                ),
                language_name=language_name,
                decoder_name=decoder_name,
                speech_duration=(
                    state["speech_duration"]
                ),
                inference_time=inference_time,
                status=status,
                error=error_text,
            )

            # ----------------------------------------------------
            # Commit result
            # ----------------------------------------------------

            if status == "ok" and text:

                state["transcript"] += (
                    f"{text}\n\n"
                )

                print(
                    f"[UTTERANCE {utterance_number}] "
                    f"COMMITTED"
                )

            elif status == "ok":

                state["transcript"] += (
                    "[No speech recognized]\n\n"
                )

                print(
                    f"[UTTERANCE {utterance_number}] "
                    f"EMPTY ASR"
                )

            else:

                state["transcript"] += (
                    "[Transcription unavailable]\n\n"
                )

            state["total_speech_seconds"] += (
                state["speech_duration"]
            )

            # ----------------------------------------------------
            # Reset
            # ----------------------------------------------------

            state["speech_active"] = False
            state["speech_duration"] = 0.0
            state["silence_duration"] = 0.0
            state["utterance_audio"] = []
            state["pre_roll_audio"] = []

            print(
                f"[UTTERANCE {utterance_number}] "
                f"STATE RESET"
            )

        # ========================================================
        # LIVE UI STATE
        # ========================================================

        if state["speech_active"]:

            status_html = ui_status(
                state,
                "speaking",
            )

        else:

            status_html = ui_status(
                state,
                "idle",
            )

        return (
            state["transcript"],
            status_html,
            state,
        )

    except Exception as e:

        print("\n[LIVE CALLBACK ERROR]")
        traceback.print_exc()

        return (
            state.get("transcript", ""),
            ui_status(state, "error"),
            state,
        )


# ============================================================
# 11. Production Gradio Application
# ============================================================

CUSTOM_CSS = """

/* ============================================================
   GLOBAL
   ============================================================ */

body {
    background:
        radial-gradient(
            circle at 15% 10%,
            rgba(59, 130, 246, 0.08),
            transparent 30%
        ),
        radial-gradient(
            circle at 85% 20%,
            rgba(139, 92, 246, 0.07),
            transparent 30%
        ),
        #090d14 !important;
}

.gradio-container {
    max-width: 1180px !important;
    margin: auto !important;
    padding: 28px 24px 40px !important;
}

/* ============================================================
   HEADER
   ============================================================ */

.app-header {
    padding: 8px 0 24px;
}

.app-eyebrow {
    font-size: 12px;
    font-weight: 700;
    letter-spacing: 0.14em;
    text-transform: uppercase;
    color: #8b9bb4;
    margin-bottom: 8px;
}

.app-title {
    font-size: 34px;
    line-height: 1.15;
    font-weight: 750;
    letter-spacing: -0.025em;
    color: #f4f7fb;
    margin: 0;
}

.app-subtitle {
    color: #8996aa;
    font-size: 15px;
    margin-top: 9px;
}

/* ============================================================
   CONFIGURATION
   ============================================================ */

.config-panel {
    background: rgba(17, 24, 39, 0.72);
    border: 1px solid rgba(148, 163, 184, 0.12);
    border-radius: 14px;
    padding: 14px;
}

/* ============================================================
   MICROPHONE
   ============================================================ */

.mic-panel {
    background:
        linear-gradient(
            145deg,
            rgba(20, 29, 43, 0.94),
            rgba(13, 18, 28, 0.94)
        );
    border: 1px solid rgba(148, 163, 184, 0.13);
    border-radius: 18px;
    padding: 20px;
    min-height: 210px;
}

.mic-heading {
    font-size: 14px;
    font-weight: 650;
    color: #dbe4f0;
    margin-bottom: 4px;
}

.mic-description {
    font-size: 13px;
    color: #7e8da4;
    margin-bottom: 15px;
}

/* ============================================================
   STATUS CARD
   ============================================================ */

.status-card {
    border-radius: 14px;
    padding: 18px;
    margin-top: 14px;
    border: 1px solid rgba(148, 163, 184, 0.12);
    background: rgba(15, 23, 35, 0.78);
}

.status-card.speaking {
    border-color: rgba(34, 197, 94, 0.22);
}

.status-card.processing {
    border-color: rgba(59, 130, 246, 0.25);
}

.status-card.error {
    border-color: rgba(239, 68, 68, 0.25);
}

.status-top {
    display: flex;
    align-items: center;
    gap: 12px;
}

.status-indicator {
    width: 10px;
    height: 10px;
    border-radius: 50%;
    background: #64748b;
    box-shadow: 0 0 0 5px rgba(100, 116, 139, 0.10);
}

.speaking .status-indicator {
    background: #22c55e;
    box-shadow: 0 0 0 5px rgba(34, 197, 94, 0.10);
}

.processing .status-indicator {
    background: #3b82f6;
    box-shadow: 0 0 0 5px rgba(59, 130, 246, 0.10);
}

.error .status-indicator {
    background: #ef4444;
    box-shadow: 0 0 0 5px rgba(239, 68, 68, 0.10);
}

.status-title {
    font-size: 15px;
    font-weight: 700;
    color: #e8eef7;
}

.status-subtitle {
    font-size: 12px;
    color: #7f8ca0;
    margin-top: 2px;
}

.status-metrics {
    display: flex;
    gap: 28px;
    margin-top: 17px;
    padding-top: 14px;
    border-top: 1px solid rgba(148, 163, 184, 0.08);
}

.metric {
    display: flex;
    flex-direction: column;
    gap: 3px;
}

.metric-label {
    font-size: 10px;
    text-transform: uppercase;
    letter-spacing: 0.09em;
    color: #6f7d92;
}

.metric-value {
    font-size: 14px;
    font-weight: 650;
    color: #dce5f1;
}

.silence-track {
    height: 4px;
    background: rgba(148, 163, 184, 0.10);
    border-radius: 999px;
    margin-top: 16px;
    overflow: hidden;
}

.silence-progress {
    height: 100%;
    border-radius: 999px;
    background: #22c55e;
    transition: width 0.25s ease;
}

.silence-caption {
    font-size: 11px;
    color: #69778c;
    margin-top: 8px;
}

.processing-line {
    margin-top: 15px;
    font-size: 13px;
    color: #8291a7;
}

/* ============================================================
   TRANSCRIPT
   ============================================================ */

.transcript-panel {
    background: rgba(13, 18, 28, 0.90);
    border: 1px solid rgba(148, 163, 184, 0.13);
    border-radius: 18px;
    overflow: hidden;
}

.transcript-header {
    padding: 16px 18px;
    border-bottom: 1px solid rgba(148, 163, 184, 0.09);
}

.transcript-title {
    font-size: 14px;
    font-weight: 700;
    color: #dce5f1;
}

.transcript-subtitle {
    font-size: 11px;
    color: #68768b;
    margin-top: 3px;
}

/* ============================================================
   FOOTER
   ============================================================ */

.system-footer {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 14px 2px 0;
    color: #566378;
    font-size: 11px;
}

.system-footer strong {
    color: #78869a;
}

/* ============================================================
   MOBILE
   ============================================================ */

@media (max-width: 700px) {

    .gradio-container {
        padding: 20px 14px 30px !important;
    }

    .app-title {
        font-size: 28px;
    }

    .status-metrics {
        gap: 18px;
    }

    .system-footer {
        flex-direction: column;
        align-items: flex-start;
        gap: 5px;
    }
}
"""


with gr.Blocks(
    title="IndicConformer • Live Transcription",
    css=CUSTOM_CSS,
    theme=gr.themes.Base(
        primary_hue="blue",
        neutral_hue="slate",
        font=[
            gr.themes.GoogleFont("Inter"),
            "ui-sans-serif",
            "system-ui",
            "sans-serif",
        ],
    ),
) as demo:

    # ========================================================
    # HEADER
    # ========================================================

    gr.HTML(
        """
        <div class="app-header">
            <div class="app-eyebrow">
                SPEECH INTELLIGENCE
            </div>

            <h1 class="app-title">
                Live Transcription
            </h1>

            <div class="app-subtitle">
                Real-time Hindi & Urdu speech recognition
                powered by IndicConformer.
            </div>
        </div>
        """
    )

    # ========================================================
    # CONFIGURATION
    # ========================================================

    with gr.Group(elem_classes="config-panel"):

        with gr.Row():

            language = gr.Dropdown(
                choices=["Hindi", "Urdu"],
                value="Hindi",
                label="Language",
                info="Recognition language",
                scale=2,
            )

            decoder = gr.Dropdown(
                choices=["RNNT"],
                value="RNNT",
                label="Decoder",
                info="Inference decoder",
                scale=2,
            )

            gr.Markdown(
                """
                **Continuous capture**

                No maximum speech duration
                """
            )

    # ========================================================
    # MAIN WORKSPACE
    # ========================================================

    with gr.Row():

        # ----------------------------------------------------
        # LEFT — MICROPHONE
        # ----------------------------------------------------

        with gr.Column(
            scale=5,
            elem_classes="mic-panel",
        ):

            gr.HTML(
                """
                <div class="mic-heading">
                    Microphone
                </div>

                <div class="mic-description">
                    Speak naturally. Your speech will be
                    automatically segmented and transcribed.
                </div>
                """
            )

            microphone = gr.Audio(
                sources=["microphone"],
                type="numpy",
                streaming=True,
                label=None,
                show_label=False,
            )

            status = gr.HTML(
                value="""
                <div class="status-card ready">
                    <div class="status-top">
                        <div class="status-indicator"></div>
                        <div>
                            <div class="status-title">
                                Ready
                            </div>
                            <div class="status-subtitle">
                                Start speaking to begin transcription
                            </div>
                        </div>
                    </div>
                </div>
                """,
            )

        # ----------------------------------------------------
        # RIGHT — TRANSCRIPT
        # ----------------------------------------------------

        with gr.Column(
            scale=7,
            elem_classes="transcript-panel",
        ):

            gr.HTML(
                """
                <div class="transcript-header">
                    <div class="transcript-title">
                        Transcript
                    </div>

                    <div class="transcript-subtitle">
                        Finalized utterances appear automatically
                    </div>
                </div>
                """
            )

            transcript = gr.Textbox(
                show_label=False,
                lines=15,
                placeholder=(
                    "Your transcription will appear here..."
                ),
                interactive=False,
            )

    # ========================================================
    # FOOTER
    # ========================================================

    gr.HTML(
        """
        <div class="system-footer">
            <span>
                <strong>IndicConformer 600M</strong>
                &nbsp;•&nbsp; 16 kHz
            </span>

            <span>
                Continuous VAD &nbsp;•&nbsp;
                Persistent session audit
            </span>
        </div>
        """
    )

    # ========================================================
    # SESSION STATE
    # ========================================================

    state = gr.State(
        make_initial_state()
    )

    # ========================================================
    # LIVE STREAM
    # ========================================================

    microphone.stream(
        fn=live_vad_asr,
        inputs=[
            microphone,
            state,
            language,
            decoder,
        ],
        outputs=[
            transcript,
            status,
            state,
        ],
        stream_every=1.0,
        show_progress="hidden",
    )


# ============================================================
# LAUNCH
# ============================================================

demo.queue(
    default_concurrency_limit=1
).launch(
    share=True,
    debug=False,
)

/tmp/ipykernel_4882/4218040561.py:753: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b66688852f803fc6e9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
# 11. Single clean Gradio application

with gr.Blocks(
    title="IndicConformer Live ASR — Clean"
) as demo:

    gr.Markdown(
        """
        # 🎙️ IndicConformer Live ASR

        **Microphone → VAD → complete utterance → IndicConformer → persistent audit**

        Speak for **any length of time**.

        The utterance is finalized only after
        **5 continuous seconds of silence**.

        CTC and RNNT remain selectable.
        """
    )

    with gr.Row():

        language = gr.Dropdown(
            choices=["Hindi", "Urdu"],
            value="Hindi",
            label="Language",
        )

        decoder = gr.Dropdown(
            choices=["RNNT"],
            value="RNNT",
            label="Decoder",
        )

    microphone = gr.Audio(
        sources=["microphone"],
        type="numpy",
        streaming=True,
        label="Microphone",
    )

    transcript = gr.Textbox(
        label="Committed Transcript / Debug",
        lines=18,
    )

    state = gr.State(
        make_initial_state()
    )

    microphone.stream(
        fn=live_vad_asr,
        inputs=[
            microphone,
            state,
            language,
            decoder,
        ],
        outputs=[
            transcript,
            state,
        ],
        stream_every=1.0,
        show_progress="hidden",
    )

demo.queue(
    default_concurrency_limit=1
).launch(
    share=True,
    debug=True,
)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://64f4269d1f533ba788.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



[AUDIO] incoming: 48000 (48000,) int16
[AUDIO] resampled 48000 → 16000
[AUDIO] normalized: 1.000s

[AUDIO] 1.00s | RMS=0.002123

[AUDIO] incoming: 48000 (48000,) int16
[AUDIO] resampled 48000 → 16000
[AUDIO] normalized: 1.000s

[AUDIO] 1.00s | RMS=0.000414

[AUDIO] incoming: 48000 (48000,) int16
[AUDIO] resampled 48000 → 16000
[AUDIO] normalized: 1.000s

[AUDIO] 1.00s | RMS=0.000370

[AUDIO] incoming: 48000 (48000,) int16
[AUDIO] resampled 48000 → 16000
[AUDIO] normalized: 1.000s

[AUDIO] 1.00s | RMS=0.002485

[AUDIO] incoming: 48000 (48000,) int16
[AUDIO] resampled 48000 → 16000
[AUDIO] normalized: 1.000s

[AUDIO] 1.00s | RMS=0.080055
🟢 SPEECH START

[AUDIO] incoming: 48000 (48000,) int16
[AUDIO] resampled 48000 → 16000
[AUDIO] normalized: 1.000s

[AUDIO] 1.00s | RMS=0.023598

[AUDIO] incoming: 48000 (48000,) int16
[AUDIO] resampled 48000 → 16000
[AUDIO] normalized: 1.000s

[AUDIO] 1.00s | RMS=0.042168

[AUDIO] incoming: 48000 (48000,) int16
[AUDIO] resampled 48000 → 16000
[AUDIO] no

In [14]:
# 12. Audit inspection
# Run this cell after stopping the live app/session.

print("Audit directory:", AUDIT_DIR)
print("Audio directory:", AUDIO_DIR)
print("JSONL:", JSONL_PATH)
print("CSV:", CSV_PATH)

if AUDIO_DIR.exists():
    wavs = sorted(AUDIO_DIR.glob("*.wav"))
    print("Saved utterances:", len(wavs))
    for path in wavs[-10:]:
        print(" -", path.name)

Audit directory: /content/asr_audit
Audio directory: /content/asr_audit/audio
JSONL: /content/asr_audit/utterance_log.jsonl
CSV: /content/asr_audit/utterance_log.csv
Saved utterances: 1
 - utterance_0001.wav


UI Upgrade